In [1]:
import pandas as pd
import numpy as np
from causalexplain import GraphDiscovery
from graphviz import Digraph
from pathlib import Path
import re

def draw_graphviz_dag(adj, nodes,labels, out_path, engine="dot"):
    #guards to check for adjacency matrix dimension
    adj = np.asarray(adj)
    print("draw_graphviz_dag adj ndim:", adj.ndim, "shape:", adj.shape)

    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]
    nodes = list(nodes)[:n]
    
    # Restore original labels where possible
    labels = labels

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    g.attr(rankdir="TB")  # left-to-right; change to "TB" if you prefer top-down
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10"
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7"
    )

    # Add nodes with restored labels
    for clean_name, label in zip(nodes, labels):
        g.node(clean_name, label=label)

    # Add edges
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)



#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_ReX"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
# Map from cleaned -> original for restoring labels in plots
def clean_and_encode_df(df: pd.DataFrame):
    """
    - Drop rows with NA.
    - Clean column names for algorithms (letters+digits, start with letter).
    - One-hot encode non-numeric columns.
    Returns:
      df_enc: encoded numeric DataFrame
      clean_to_orig: dict {clean_name: original_name}
    """
    df = df.dropna().copy()
    if df.empty:
        raise ValueError("Data frame is empty after dropna().")

    # 1) Clean base column names
    orig_cols = list(df.columns)
    clean_cols = []
    for c in orig_cols:
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)  # remove underscores, spaces, etc.
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        clean_cols.append(c2)
    df.columns = clean_cols
    clean_to_orig = dict(zip(clean_cols, orig_cols))

    # 2) One-hot encode non-numeric columns
    non_numeric = df.select_dtypes(exclude=["number"]).columns
    if len(non_numeric) > 0:
        df_enc = pd.get_dummies(df, columns=list(non_numeric), drop_first=False, dtype=float)
    else:
        df_enc = df.astype(float)

    return df_enc, clean_to_orig

In [3]:
import networkx as nx
import time
def run_rex(df, experiment_name="rex_exp"):
    #Graph Discovery expects a file path for dataframe
    tmp_csv = "tmp_rex_input.csv"
    df = clean_name(df)
    df.to_csv(tmp_csv, index=False) #write df to a csv temporarily

    #init graph discovery instance
    start = time.time()
    gd = GraphDiscovery(experiment_name=experiment_name, model_type="rex",csv_filename= tmp_csv)

    gd.run(quiet=True)
    end=time.time()

    dot_path = output_dir / f"{experiment_name}.dot"
    gd.export_dag(str(dot_path))

    # Read DOT with networkx and build adjacency
    adj, nodes = dot_to_adjacency(dot_path)
    print("ReX adj ndim:", adj.ndim, "shape:", adj.shape)
    print(f"ReX took {(end - start)/60:.2f} minutes")

    return adj, nodes

In [4]:
def clean_name(df):
    #makes column names ReX friendly i.e start with letter, contain only letters and numbers
    new_cols = []
    for c in df.columns:
        # remove underscores and other non-alphanumeric
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)
        # if it doesn't start with a letter, prefix with 'X'
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        new_cols.append(c2)
    df2 = df.copy()
    df2.columns = new_cols
    return df2

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [5]:
def dot_to_adjacency(dot_path):
    G = nx.drawing.nx_pydot.read_dot(str(dot_path))
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    p = len(nodes)
    adj = np.zeros((p, p), dtype=int)
    for u, v in G.edges():
        i = idx[u]
        j = idx[v]
        adj[i, j] = 1

    adj = np.asarray(adj)
    if adj.ndim == 1:              # safety
        adj = adj.reshape(1, 1)
    return adj, nodes

In [6]:
csv_path = output_dir/ "NIJ_graph_ReX_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        df_subset_enc = encode_mixed_df(df_subset)
        adj, _ = run_rex(df_subset_enc)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)
def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))
    return(frac*100)

In [7]:
csv_path = output_dir/ "NIJ_graph_ReX_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

seeds = [42,7,12]
incompat_scores = []
disagree = []
for seed in seeds:
    score = incompatibility_score(A, k = 5, n_subsets=10, seed=seed)
    incompat_scores.append(score)
    disagree.append(goodness(score, 5))
    

Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 26.22 minutes
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 22.01 minutes
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 25.48 minutes
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 22.53 minutes
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 17.88 minutes
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (5, 5)
ReX took 26.26 minute

In [9]:
incompat_scores
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 0.47842333648024393
standard dev of disagreement percentage: 2.3921166824012206


In [10]:
incompat_scores

In [11]:
disagree